In [1]:
import pandas as pd
import pickle
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn_quantile import RandomForestQuantileRegressor
from tqdm.auto import tqdm

import sys
sys.path.append('../../../TaskExecutionTimeMining/')
from quantile_regression import QuantileRegression


In [2]:
with open('../../transformed_event_logs/Helpdesk_train.pickle', 'rb') as f:
    train_data = pickle.load(f)

In [3]:
train_data

,Case ID,Activity_start,Resource_start,Complete Timestamp_start,Variant_start,Variant index_start,Variant.1_start,seriousness_start,customer_start,product_start,...,intercase_n_3__Wait_Resolve ticket_Wait,intercase_n_3__Wait_Take in charge ticket_Create SW anomaly,intercase_n_3__Wait_Take in charge ticket_Require upgrade,intercase_n_3__Wait_Take in charge ticket_Resolve ticket,intercase_n_3__Wait_Take in charge ticket_Take in charge ticket,intercase_n_3__Wait_Take in charge ticket_Wait,intercase_n_3__Wait_Wait_Create SW anomaly,intercase_n_3__Wait_Wait_Resolve ticket,intercase_n_3__Wait_Wait_Take in charge ticket,intercase_n_3__Wait_Wait_Wait
0,Case 1,Assign seriousness,Value 1,2012-10-09 14:50:17,Variant 12,12,Variant 12,Value 1,Value 1,Value 1,...,0,0,0,0,0,0,0,0,0,0
1,Case 1,Take in charge ticket,Value 1,2012-10-09 14:51:01,Variant 12,12,Variant 12,Value 1,Value 1,Value 1,...,0,0,0,0,0,0,0,0,0,0
2,Case 1,Take in charge ticket,Value 2,2012-10-12 15:02:56,Variant 12,12,Variant 12,Value 1,Value 1,Value 1,...,0,0,0,0,0,1,0,0,0,0
3,Case 1,Resolve ticket,Value 1,2012-10-25 11:54:26,Variant 12,12,Variant 12,Value 1,Value 1,Value 1,...,0,0,0,0,0,1,0,0,0,0
4,Case 2,Assign seriousness,Value 4,2012-04-03 08:55:38,Variant 1,1,Variant 1,Value 1,Value 2,Value 2,...,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16750,Case 4579,Take in charge ticket,Value 9,2010-07-26 13:31:59,Variant 1,1,Variant 1,Value 1,Value 71,Value 3,...,0,0,0,0,0,2,0,0,0,0
16751,Case 4579,Resolve ticket,Value 9,2010-07-26 13:32:11,Variant 1,1,Variant 1,Value 1,Value 71,Value 3,...,0,0,0,0,0,2,0,0,0,0
16752,Case 4580,Take in charge ticket,Value 6,2012-01-03 09:33:43,Variant 18,18,Variant 18,Value 1,Value 92,Value 3,...,0,0,0,0,0,0,0,0,0,0
16753,Case 4580,Wait,Value 6,2012-01-10 15:30:11,Variant 18,18,Variant 18,Value 1,Value 92,Value 3,...,0,0,0,0,0,0,0,0,0,0


In [4]:
list(train_data.columns)

['Case ID',
 'Activity_start',
 'Resource_start',
 'Complete Timestamp_start',
 'Variant_start',
 'Variant index_start',
 'Variant.1_start',
 'seriousness_start',
 'customer_start',
 'product_start',
 'responsible_section_start',
 'seriousness_2_start',
 'service_level_start',
 'service_type_start',
 'support_section_start',
 'workgroup_start',
 'id_start',
 'Activity_complete',
 'Resource_complete',
 'Complete Timestamp_complete',
 'Variant_complete',
 'Variant index_complete',
 'Variant.1_complete',
 'seriousness_complete',
 'customer_complete',
 'product_complete',
 'responsible_section_complete',
 'seriousness_2_complete',
 'service_level_complete',
 'service_type_complete',
 'support_section_complete',
 'workgroup_complete',
 'id_complete',
 'duration',
 'duration_seconds',
 'duration_ms',
 'duration_hours',
 'seconds_in_day',
 'day_of_week',
 'Assign seriousness',
 'Closed',
 'Create SW anomaly',
 'DUPLICATE',
 'INVALID',
 'Insert ticket',
 'RESOLVED',
 'Require upgrade',
 'Resol

In [5]:
activity_count = [
    'Assign seriousness',
    'Closed',
    'Create SW anomaly',
    'DUPLICATE',
    'INVALID',
    'Insert ticket',
    'RESOLVED',
    'Require upgrade',
    'Resolve SW anomaly',
    'Resolve ticket',
    'Schedule intervention',
    'Take in charge ticket',
    'VERIFIED',
    'Wait'
 ]

resource_count = [
 'Value 1',
 'Value 10',
 'Value 11',
 'Value 12',
 'Value 13',
 'Value 14',
 'Value 15',
 'Value 16',
 'Value 17',
 'Value 18',
 'Value 19',
 'Value 2',
 'Value 20',
 'Value 21',
 'Value 22',
 'Value 3',
 'Value 4',
 'Value 5',
 'Value 6',
 'Value 7',
 'Value 8',
 'Value 9',
]

ii1 = [
   'intercase_n_1__Assign seriousness',
 'intercase_n_1__Closed',
 'intercase_n_1__Create SW anomaly',
 'intercase_n_1__DUPLICATE',
 'intercase_n_1__INVALID',
 'intercase_n_1__Insert ticket',
 'intercase_n_1__RESOLVED',
 'intercase_n_1__Require upgrade',
 'intercase_n_1__Resolve SW anomaly',
 'intercase_n_1__Resolve ticket',
 'intercase_n_1__Schedule intervention',
 'intercase_n_1__Take in charge ticket',
 'intercase_n_1__VERIFIED',
 'intercase_n_1__Wait', 
]

ii3 = [
    'intercase_n_3__Assign seriousness',
 'intercase_n_3__Assign seriousness_Assign seriousness',
 'intercase_n_3__Assign seriousness_Assign seriousness_Assign seriousness',
 'intercase_n_3__Assign seriousness_Assign seriousness_Resolve ticket',
 'intercase_n_3__Assign seriousness_Assign seriousness_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Assign seriousness_Wait',
 'intercase_n_3__Assign seriousness_Create SW anomaly',
 'intercase_n_3__Assign seriousness_Create SW anomaly_Create SW anomaly',
 'intercase_n_3__Assign seriousness_Create SW anomaly_Require upgrade',
 'intercase_n_3__Assign seriousness_Create SW anomaly_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Require upgrade',
 'intercase_n_3__Assign seriousness_Require upgrade_Require upgrade',
 'intercase_n_3__Assign seriousness_Require upgrade_Resolve ticket',
 'intercase_n_3__Assign seriousness_Resolve ticket',
 'intercase_n_3__Assign seriousness_Resolve ticket_Closed',
 'intercase_n_3__Assign seriousness_Resolve ticket_Resolve ticket',
 'intercase_n_3__Assign seriousness_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Resolve ticket_Wait',
 'intercase_n_3__Assign seriousness_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Assign seriousness',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Require upgrade',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Resolve SW anomaly',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Schedule intervention',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Take in charge ticket_Wait',
 'intercase_n_3__Assign seriousness_Wait',
 'intercase_n_3__Assign seriousness_Wait_Assign seriousness',
 'intercase_n_3__Assign seriousness_Wait_Resolve ticket',
 'intercase_n_3__Assign seriousness_Wait_Take in charge ticket',
 'intercase_n_3__Assign seriousness_Wait_Wait',
 'intercase_n_3__Closed_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Create SW anomaly',
 'intercase_n_3__Create SW anomaly_Create SW anomaly_Resolve SW anomaly',
 'intercase_n_3__Create SW anomaly_Create SW anomaly_Resolve ticket',
 'intercase_n_3__Create SW anomaly_Require upgrade_Require upgrade',
 'intercase_n_3__Create SW anomaly_Require upgrade_Resolve ticket',
 'intercase_n_3__Create SW anomaly_Require upgrade_VERIFIED',
 'intercase_n_3__Create SW anomaly_Resolve SW anomaly',
 'intercase_n_3__Create SW anomaly_Resolve SW anomaly_Resolve SW anomaly',
 'intercase_n_3__Create SW anomaly_Resolve SW anomaly_Resolve ticket',
 'intercase_n_3__Create SW anomaly_Resolve ticket_RESOLVED',
 'intercase_n_3__Create SW anomaly_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Create SW anomaly_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Create SW anomaly_Take in charge ticket_Wait',
 'intercase_n_3__Insert ticket',
 'intercase_n_3__Insert ticket_Assign seriousness',
 'intercase_n_3__Insert ticket_Assign seriousness_Assign seriousness',
 'intercase_n_3__Insert ticket_Assign seriousness_Resolve ticket',
 'intercase_n_3__Insert ticket_Assign seriousness_Take in charge ticket',
 'intercase_n_3__Insert ticket_Take in charge ticket',
 'intercase_n_3__Insert ticket_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Insert ticket_Wait',
 'intercase_n_3__Insert ticket_Wait_Wait',
 'intercase_n_3__RESOLVED_INVALID_Closed',
 'intercase_n_3__RESOLVED_INVALID_VERIFIED',
 'intercase_n_3__Require upgrade_Create SW anomaly_Resolve ticket',
 'intercase_n_3__Require upgrade_Require upgrade_Create SW anomaly',
 'intercase_n_3__Require upgrade_Require upgrade_Require upgrade',
 'intercase_n_3__Require upgrade_Require upgrade_Resolve ticket',
 'intercase_n_3__Require upgrade_Require upgrade_Take in charge ticket',
 'intercase_n_3__Require upgrade_Resolve ticket_Resolve ticket',
 'intercase_n_3__Require upgrade_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Require upgrade_Take in charge ticket_Wait',
 'intercase_n_3__Require upgrade_VERIFIED_DUPLICATE',
 'intercase_n_3__Require upgrade_Wait_Resolve ticket',
 'intercase_n_3__Resolve SW anomaly_Require upgrade_Create SW anomaly',
 'intercase_n_3__Resolve SW anomaly_Require upgrade_Resolve ticket',
 'intercase_n_3__Resolve SW anomaly_Resolve SW anomaly_Require upgrade',
 'intercase_n_3__Resolve SW anomaly_Resolve SW anomaly_Resolve ticket',
 'intercase_n_3__Resolve ticket',
 'intercase_n_3__Resolve ticket_Assign seriousness_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Closed_Take in charge ticket',
 'intercase_n_3__Resolve ticket_RESOLVED_INVALID',
 'intercase_n_3__Resolve ticket_Require upgrade_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Resolve ticket_Require upgrade',
 'intercase_n_3__Resolve ticket_Resolve ticket_Resolve ticket',
 'intercase_n_3__Resolve ticket_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Require upgrade',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Resolve ticket_Take in charge ticket_Wait',
 'intercase_n_3__Resolve ticket_Wait_Resolve ticket',
 'intercase_n_3__Resolve ticket_Wait_Wait',
 'intercase_n_3__Schedule intervention_Take in charge ticket_Wait',
 'intercase_n_3__Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Assign seriousness_Assign seriousness',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Create SW anomaly',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Require upgrade',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Resolve SW anomaly',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Create SW anomaly_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Require upgrade_Create SW anomaly',
 'intercase_n_3__Take in charge ticket_Require upgrade_Require upgrade',
 'intercase_n_3__Take in charge ticket_Require upgrade_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Require upgrade_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Require upgrade_Wait',
 'intercase_n_3__Take in charge ticket_Resolve SW anomaly_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Assign seriousness',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Closed',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Resolve ticket_Wait',
 'intercase_n_3__Take in charge ticket_Schedule intervention_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Schedule intervention_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Require upgrade',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Schedule intervention',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Take in charge ticket_Wait',
 'intercase_n_3__Take in charge ticket_Wait',
 'intercase_n_3__Take in charge ticket_Wait_Create SW anomaly',
 'intercase_n_3__Take in charge ticket_Wait_Require upgrade',
 'intercase_n_3__Take in charge ticket_Wait_Resolve ticket',
 'intercase_n_3__Take in charge ticket_Wait_Take in charge ticket',
 'intercase_n_3__Take in charge ticket_Wait_Wait',
 'intercase_n_3__VERIFIED_DUPLICATE_Resolve ticket',
 'intercase_n_3__Wait',
 'intercase_n_3__Wait_Assign seriousness_Assign seriousness',
 'intercase_n_3__Wait_Assign seriousness_Take in charge ticket',
 'intercase_n_3__Wait_Create SW anomaly_Require upgrade',
 'intercase_n_3__Wait_Create SW anomaly_Resolve ticket',
 'intercase_n_3__Wait_Require upgrade_Require upgrade',
 'intercase_n_3__Wait_Require upgrade_Resolve ticket',
 'intercase_n_3__Wait_Require upgrade_Wait',
 'intercase_n_3__Wait_Resolve ticket',
 'intercase_n_3__Wait_Resolve ticket_Resolve ticket',
 'intercase_n_3__Wait_Resolve ticket_Take in charge ticket',
 'intercase_n_3__Wait_Resolve ticket_Wait',
 'intercase_n_3__Wait_Take in charge ticket_Create SW anomaly',
 'intercase_n_3__Wait_Take in charge ticket_Require upgrade',
 'intercase_n_3__Wait_Take in charge ticket_Resolve ticket',
 'intercase_n_3__Wait_Take in charge ticket_Take in charge ticket',
 'intercase_n_3__Wait_Take in charge ticket_Wait',
 'intercase_n_3__Wait_Wait_Create SW anomaly',
 'intercase_n_3__Wait_Wait_Resolve ticket',
 'intercase_n_3__Wait_Wait_Take in charge ticket',
 'intercase_n_3__Wait_Wait_Wait'
]

feature_combinations = {
        'A' : ['Activity_start'],
        'R' : ['Resource_start'],
        'AR' : ['Activity_start', 'Resource_start'],
        'ARS' : ['Activity_start', 'Resource_start', 'seconds_in_day'],
        #'ASAC' : ['Activity_start', 'seconds_in_day'] + activity_count,
        #'RSRC' : ['Resource_start', 'seconds_in_day'] + resource_count,
        'ARSAC' : ['Activity_start', 'Resource_start', 'seconds_in_day'] + activity_count,
        'ARCRC' : ['Activity_start', 'Resource_start'] + resource_count,
        #'ARACRC' : ['Activity_start', 'Resource_start'] + activity_count + resource_count,
        'ARSACRC' : ['Activity_start', 'Resource_start', 'seconds_in_day'] + activity_count + resource_count,
        'ARSD' : ['Activity_start', 'Resource_start', 'seconds_in_day', 'day_of_week'],
        'ARSDACRC' : ['Activity_start', 'Resource_start', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count,
        'ARSDII1' : ['Activity_start', 'Resource_start', 'seconds_in_day', 'day_of_week'] + ii1,
        'ARSDII3' : ['Activity_start', 'Resource_start', 'seconds_in_day', 'day_of_week'] + ii3,
        'ARSDACRCII1' : ['Activity_start', 'Resource_start', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count + ii1,
        'ARSDACRCII3' : ['Activity_start', 'Resource_start', 'seconds_in_day', 'day_of_week'] + activity_count + resource_count + ii3

}

In [6]:
quantile_regression_models = {}
for k, v in tqdm(feature_combinations.items()):
    qrm = QuantileRegression(train_data, v)
    qrm.fit()
    quantile_regression_models[k] = qrm

  0%|          | 0/13 [00:00<?, ?it/s]

In [7]:
out_path = './quantile_regression_models.pkl'
with open(out_path, 'wb') as out_file:
    pickle.dump(quantile_regression_models, out_file, protocol=pickle.HIGHEST_PROTOCOL)